# Notebook 3: Attention-Analyse
**Kurs:** Mechanistic Interpretability  
**Modell:** EleutherAI/pythia-410m  
**Ziel:** Attention-Muster visualisieren und wichtige Heads identifizieren.

> **Aufgabe:** Implementiere die Zellen schrittweise. Hilfreiche Dokumentation:
> - HuggingFace `output_attentions`: https://huggingface.co/docs/transformers/main_classes/output
> - Matplotlib Imshow: https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.imshow.html
> - NumPy argsort: https://numpy.org/doc/stable/reference/generated/numpy.argsort.html

## Hintergrund: Attention-Mechanismus

In jedem Transformer-Layer berechnen **Attention Heads** gewichtete Summen über alle Token-Positionen:

```
A = softmax( QK^T / sqrt(d_head) )
```

Die Attention-Matrix A hat die Form **(seq × seq)**:
- Zeile i: Welche Positionen achtet Position i?
- Spalte j: Von welchen Positionen wird Position j beachtet?

In der Mechanistic Interpretability fragen wir: **Welche Heads implementieren welche Funktion?**

Bekannte Head-Typen:
- **Previous Token Heads**: Achten auf das unmittelbar vorherige Token
- **Induction Heads**: Implementieren Muster-Vervollständigung ("A...B...A → B")
- **Name Mover Heads**: Transportieren semantische Information zur Ausgabe-Position

## 1. Setup und Modell laden

In [ ]:
# TODO: Importiere torch, numpy, matplotlib
# TODO: Lade Tokenizer und Modell (wie in Notebook 1)
#
# MODEL_NAME = "EleutherAI/pythia-410m"
# n_layers = model.config.num_hidden_layers
# n_heads  = model.config.num_attention_heads

## 2. Forward Pass mit Attention-Gewichten

In [ ]:
prompt = "The capital of France is Paris, and the capital of Germany is"

# TODO: Tokenisiere den Prompt und konvertiere zu Token-Strings
# TODO: Führe einen Forward Pass mit output_attentions=True durch
#
#   outputs = model(**inputs, output_attentions=True)
#
# TODO: Analysiere outputs.attentions:
#   - Wie viele Attention-Matrizen gibt es?
#   - Was ist die Shape jeder Matrix? (Tipp: batch, heads, seq, seq)
#
# Hinweis: outputs.attentions ist ein Tupel der Länge n_layers

## 3. Einzelne Attention-Heads visualisieren

In [ ]:
# TODO: Implementiere plot_attention_head(attentions, layer, head, tokens):
#   1. Extrahiere die Attention-Matrix: attentions[layer][0, head]
#   2. Konvertiere zu NumPy: .cpu().numpy()
#   3. Erstelle Heatmap mit plt.imshow(..., cmap="Blues")
#   4. Setze Token-Labels auf X- und Y-Achse
#   5. Beschrifte Achsen ("Key (Quelle)" / "Query (Ziel)")
#
# TODO: Visualisiere 6 verschiedene Heads in einem 2×3-Grid (plt.subplots(2, 3))
#   Wähle unterschiedliche Schichten (z.B. 0, 5, 10, 15, 20, 23) und Heads
#
# Dokumentation plt.subplots:
#   https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html

## 4. Attention auf Subject-Token

In [ ]:
# Ziel: Wie stark achtet die LETZTE Position auf das Subject-Token "France"?
# Dies gibt einen Hinweis auf "Faktenwissen-Abruf"-Heads.
#
# TODO: Finde die Position des Tokens "France" in der Token-Liste
#   Tipp: tokens.index(subject_token) oder verwende eine List-Comprehension
#
# TODO: Berechne für ALLE Layers und Heads:
#   att_to_subject[layer, head] = attentions[layer][0, head, last_pos, subject_pos]
#   Ergebnis: Matrix der Form (n_layers, n_heads)
#
# Tipp: Zwei verschachtelte for-Schleifen (layer, head)
# Hinweis: np.zeros((n_layers, n_heads)) erstellt eine Nullmatrix

## 5. Heatmap: Alle Heads und Layers

In [ ]:
# TODO: Erstelle eine Heatmap mit plt.imshow() für die att_to_subject-Matrix:
#   - Y-Achse: Schichten
#   - X-Achse: Heads
#   - Farbe: Attention-Gewicht (cmap="hot" oder "Blues")
#   - Colorbar hinzufügen: plt.colorbar(im, ...)
#
# TODO: Markiere die Top-5 Heads mit einem Rechteck (plt.Rectangle)
#   Tipp: np.argsort(att_to_subject.ravel())[-5:] gibt die 5 höchsten Indizes
#   Tipp: np.unravel_index(idx, (n_layers, n_heads)) konvertiert flachen Index
#
# Dokumentation plt.Rectangle:
#   https://matplotlib.org/stable/api/patches_api.html#matplotlib.patches.Rectangle

## 6. Top-Heads-Rangliste

In [ ]:
# TODO: Erstelle eine Rangliste der Top-10 Heads nach att_to_subject-Wert
#   Ausgabe: Rang | Layer | Head | Attention-Gewicht
#
# Tipp: np.argsort(att_to_subject.ravel())[::-1] sortiert absteigend
# Tipp: divmod(flat_idx, n_heads) gibt (layer, head) zurück

## 7. BONUS: Induction Heads suchen

In [ ]:
# BONUS-AUFGABE: Induction Heads erkennt man an einem charakteristischen Muster:
#   Sie achten bei Token i auf Token (i-2) oder folgen einem "A...B...A → B"-Muster.
#
# TODO: Implementiere eine induction_score(att_matrix) Funktion:
#   Misst, wie stark eine Attention-Matrix auf die um 1-2 Positionen versetzte
#   Diagonale konzentriert ist.
#
# TODO: Berechne den Score für alle Layers/Heads und erstelle eine Heatmap.
# TODO: Welcher Head hat den höchsten Induction Score?
#
# Weiterführende Lektüre:
#   Olsson et al. (2022): "In-context Learning and Induction Heads"
#   https://arxiv.org/abs/2209.11895

## 8. Reflexionsfragen

In [ ]:
# Beantworte folgende Fragen:
#
# 1. Welcher Head (Layer, Head-Nummer) achtet am stärksten auf "France"?
# 2. In welchen Schichten-Bereichen findest du die stärkste Subject-Attention?
# 3. Warum reicht Attention-Analyse allein NICHT aus, um Kausalität zu zeigen?
# 4. Was ist der Unterschied zwischen Attention-Analyse und Activation Patching?
#
# Denke daran: Hohe Attention ≠ kausale Wichtigkeit\!
# (Das behandeln wir in Notebook 5 mit Activation Patching)